# Forward rendering

jaxCAD's image renderer favors predictable forward rendering: hard visibility, early-exit sphere tracing, finite-difference normals, GGX materials, geometric AO, soft shadows, reflections, and refraction. Geometry and constraints remain JAX-native and differentiable; rendered pixels are deliberately not an optimization interface.

In [ ]:
import matplotlib.pyplot as plt

from jaxcad.render import Camera, Material, RenderSettings, Scene, render_scene
from jaxcad.sdf.boolean import Union
from jaxcad.sdf.primitives import Plane, Sphere
from jaxcad.sdf.transforms import Translate

In [ ]:
geometry = Union(
    Translate(
        Sphere(0.9, material=Material(color=[0.8, 0.12, 0.08], roughness=0.5)), [-1.0, 0.0, 0.0]
    ),
    Translate(
        Sphere(0.9, material=Material(color=[0.9, 0.65, 0.15], roughness=0.18, metallic=0.9)),
        [1.0, 0.0, 0.0],
    ),
    Plane(-0.9, material=Material(color=[0.22, 0.25, 0.3], roughness=0.8)),
    smoothness=0.0,
)
scene = Scene(
    geometry,
    camera=Camera(position=(4.5, 2.8, 7.0), target=(0.0, -0.1, 0.0), fov=0.55),
    light_directions=((0.6, 1.0, 0.4), (-0.5, 0.4, -0.2)),
    light_colors=((1.0, 0.9, 0.75), (0.25, 0.35, 0.6)),
)

In [ ]:
settings = RenderSettings.balanced((240, 320))
image = render_scene(scene, settings)
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis("off");

## Quality presets

The named presets make the performance/fidelity trade-off explicit. They are immutable dataclasses, so use `dataclasses.replace` for scene-specific changes.

In [ ]:
presets = [
    RenderSettings.draft((180, 240)),
    RenderSettings.balanced((180, 240)),
    RenderSettings.high_quality((180, 240)),
]
images = [render_scene(scene, preset) for preset in presets]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, label, rendered in zip(axes, ["Draft", "Balanced", "High quality"], images):
    axis.imshow(rendered)
    axis.set_title(label)
    axis.axis("off")
plt.tight_layout()